# Dataset Statistics — Sample Grid and Split Counts

Paper-facing dataset-side notebook. Produces Fig. 2 (sample grid) and Table 2 (split counts) for the NeurIPS 2026 Evaluations and Datasets Track submission.

Source of truth for paper structure: `../../../passage-paper/neurips/skeleton.md` (v3). Do not change figure or table identifiers here without updating the skeleton.

## Paper Mapping

| Claim | `\evidence{}` tag | Output artifact |
| --- | --- | --- |
| Sec. 3.3 sample grid across the seven resolutions | `passage/paper/notebooks/dataset_statistics.ipynb -> passage/paper/figures/sample_grid.pdf` | `figures/sample_grid.pdf` |
| Sec. 3.3 Table 2 generated split counts | `passage/paper/notebooks/dataset_statistics.ipynb -> dataset_statistics.csv` | `figures/dataset_statistics.csv` |

## Inputs / Outputs

| Kind | Path |
| --- | --- |
| Input | One generated Parquet shard per resolution under `../../outputs/export/` |
| Input | Geographic hold-out mask at `../../config/split_mask.npy` |
| Input | Release manifest `../../outputs/export/manifest.sha256` |
| Output | `../figures/sample_grid.pdf` (+ `.png`) |
| Output | `../figures/dataset_statistics.csv` |
| Output | `../figures/dataset_statistics.manifest.json` |


In [ ]:
# Setup: import helpers and handle optional dependencies.
from __future__ import annotations

import sys
from pathlib import Path

PAPER_DIR = Path.cwd().resolve().parent
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

try:
    from helpers import FIGURES_DIR, emit_manifest, save_figure
except ImportError as exc:  # pragma: no cover
    raise ImportError(
        "Could not import passage.paper.helpers; run from passage/paper/notebooks/."
    ) from exc

try:
    import numpy as np
    import matplotlib.pyplot as plt
    HAS_PLOTTING = True
except ImportError:
    HAS_PLOTTING = False

try:
    import passage  # noqa: F401
    HAS_PASSAGE = True
except ImportError:
    HAS_PASSAGE = False

STUDY_ID = "dataset_statistics"
RESOLUTIONS = [64, 128, 256, 512, 1024, 2048, 4096]
SPLITS = ["train", "calibration", "validation", "test"]

In [ ]:
# Build the per-resolution split-count table (Table 2).
#
# TODO: wire this to the release manifest at
# passage/outputs/export/manifest.sha256 and to the split mask at
# passage/config/split_mask.npy. For now we emit a stub table whose values
# are placeholders so the notebook remains runnable in a fresh kernel
# without the data share.

records = []
for res in RESOLUTIONS:
    row = {"resolution": res}
    for split in SPLITS:
        row[split] = None  # replaced by the real manifest sweep
    row["total"] = None
    records.append(row)

csv_path = FIGURES_DIR / f"{STUDY_ID}.csv"
header = ["resolution", *SPLITS, "total"]
lines = [",".join(header)]
for row in records:
    lines.append(",".join("" if row[k] is None else str(row[k]) for k in header))
csv_path.write_text("\n".join(lines) + "\n")
print(f"wrote {csv_path}")

In [ ]:
# Build the 4xR sample grid (Fig. 2).
#
# Rows: elevation, markers, obstacles, A* path overlay.
# Columns: one per resolution in RESOLUTIONS.
#
# TODO: replace the placeholder arrays with real loaded tiles from
# passage.export. Notebook remains runnable with stub data so the
# artifact contract still holds when the dataset share is absent.

if HAS_PLOTTING:
    fig, axes = plt.subplots(
        4, len(RESOLUTIONS), figsize=(2.0 * len(RESOLUTIONS), 8.0), squeeze=False
    )
    row_labels = ["elevation", "markers", "obstacles", "A* path"]
    for ci, res in enumerate(RESOLUTIONS):
        rng = np.random.default_rng(seed=42 + ci)
        for ri, label in enumerate(row_labels):
            ax = axes[ri, ci]
            ax.imshow(rng.random((min(res, 128), min(res, 128))), cmap="gray")
            ax.set_xticks([])
            ax.set_yticks([])
            if ri == 0:
                ax.set_title(f"{res}x{res}", fontsize=8)
            if ci == 0:
                ax.set_ylabel(label, fontsize=8)
    fig.suptitle("PASSAGE sample grid (stub data — replace with real tiles)")
    fig.tight_layout()
else:
    fig = None

In [ ]:
# Emit manifest and save figure.
outputs = [csv_path]

if fig is not None:
    # The paper refers to this asset as sample_grid.pdf; we keep that stem.
    pdf_path = save_figure(fig, study_id="sample_grid")
    outputs.extend([pdf_path, pdf_path.with_suffix(".png")])

manifest_path = emit_manifest(
    study_id=STUDY_ID,
    inputs={
        "resolutions": RESOLUTIONS,
        "splits": SPLITS,
        "has_plotting": HAS_PLOTTING,
        "has_passage_package": HAS_PASSAGE,
        "paper_section": "3.3",
        "paper_anchors": ["fig:sample-grid", "tab:split-counts"],
    },
    outputs=outputs,
)
print(f"wrote {manifest_path}")